In [ ]:
# ============================================================================
# INSTALLATION AND IMPORTS
# ============================================================================

!pip install -q gymnasium simpy

import random
import statistics

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import simpy
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm

from collections import defaultdict, deque
from scipy import stats
from torch.distributions import Categorical


# ============================================================================
# GLOBAL PARAMETERS
# ============================================================================

# Number of simulated days in each episode.
NUM_STEPS = 150

# Each product can receive an order between 0 and MAX_ORDER_QTY.
MAX_ORDER_QTY = 30

# Number of possible quantities for one product:
# 0, 1, 2, ..., MAX_ORDER_QTY.
ACTION_SIZE = MAX_ORDER_QTY + 1

# The two product decisions are encoded as one discrete action.
TOTAL_ACTIONS = ACTION_SIZE * ACTION_SIZE

# Scaling factor used to normalize neural-network inputs.
STATE_SCALE = 50.0

# Seeds used for training and evaluation.
seeds = [
    6145, 4098, 6148, 2055, 4103, 8200, 2058, 2062, 6159, 6161,
    18, 2067, 20, 4117, 2039, 4115, 6164, 24, 4123, 4124,
    6173, 29, 8221, 2081, 6177, 40, 4137, 42, 45, 6189,
    8240, 2098, 4146, 8249, 8252, 8253, 4159, 8259, 4164,
    6213, 4168, 72, 6220, 2124, 8270, 8268, 80, 4178, 6229,
    8278, 87, 4185, 2139, 92, 8283, 8286, 4192, 8290, 6245,
    4198, 104, 2153, 4207, 6257, 2161, 113, 4210, 4213,
    4214, 8311, 120, 6265, 4219, 126, 2175, 6274, 4226,
    6275, 4232, 2186, 138, 2189, 4240, 145, 8338, 2196,
    149, 8342, 8343, 2200, 6297, 154, 155, 156, 8350, 2209,
    8356, 2213, 8358, 165, 2216, 2218, 6317, 6318, 6320,
    177, 4274, 8371, 8372, 2230, 6327, 184, 186, 4284,
    2239, 2242, 2243, 6340, 6346, 4305, 6353, 8411, 220,
    8413, 8414, 6368, 6370, 6374, 4327, 4329, 6378, 2284,
    237, 4336, 6384, 244, 2295, 6392, 2298, 8444, 252,
    6396, 2304, 4353, 259, 6406, 8456, 2313, 4363, 8460,
    270, 6416, 273, 2322, 2323, 6417, 6426, 6430, 8480,
    2338, 293, 2342, 8487, 4392, 8491, 300, 4396, 4398
]


# ============================================================================
# ACTION ENCODING AND DECODING
# ============================================================================

def action_to_pair(
    action: int,
    action_size: int = ACTION_SIZE
) -> tuple[int, int\]:
    """
    Convert one combined action into two order quantities.

    With MAX_ORDER_QTY = 30:
        action is between 0 and 960;
        each order quantity is between 0 and 30.
    """

    order_product_1 = action // action_size
    order_product_2 = action % action_size

    return order_product_1, order_product_2


def pair_to_action(
    order_product_1: int,
    order_product_2: int,
    action_size: int = ACTION_SIZE
) -> int:
    """
    Encode two order quantities as one discrete action.
    """

    return order_product_1 * action_size + order_product_2


# ============================================================================
# PRODUCT CONTROLLED BY THE S-MIN/S-MAX POLICY
# ============================================================================

class Product:
    """
    Simulate one product controlled by a periodic-review (s, S) policy.

    The policy observes the inventory position:

        inventory position = on-hand inventory + outstanding orders

    If the inventory position is below s, an order is placed to restore
    the inventory position to S.
    """

    def __init__(
        self,
        env: simpy.Environment,
        demand_rate: float = 0.1,
        demand_distribution=(
            [1, 2, 3, 4],
            [1/6, 1/3, 1/3, 1/6]
        ),
        inventory_level: float = 60,
        inventory_check_interval: float = 1,
        lead_time_min: float = 0.5,
        lead_time_max: float = 1.0,
        s_max: float = 40,
        s_min: float = 20,
        order_setup_cost: float = 10,
        order_incremental_cost: float = 3,
        holding_cost: float = 1,
        shortage_cost: float = 7
    ):
        self.env = env

        # The demand_rate parameter is the rate lambda of the exponential
        # inter-arrival-time distribution.
        self.demand_rate = demand_rate
        self.demand_distribution = demand_distribution

        self._inventory_level = float(inventory_level)
        self.inventory_check_interval = inventory_check_interval

        self.lead_time_min = lead_time_min
        self.lead_time_max = lead_time_max

        self.s_max = s_max
        self.s_min = s_min

        self.order_setup_cost = order_setup_cost
        self.order_incremental_cost = order_incremental_cost
        self.holding_cost = holding_cost
        self.shortage_cost = shortage_cost

        # Total quantity already ordered but not yet delivered.
        self.on_order = 0.0

        self.total_order_cost = 0.0

        # inventory_history maps an inventory level to the total duration
        # spent at that level.
        self.inventory_history = defaultdict(float)

        # History used for plotting the inventory trajectory.
        self.inventory_trajectory = [
            (0.0, self._inventory_level)
        ]

        self.last_inventory_level = self._inventory_level
        self.last_inventory_update_time = 0.0

        self.last_step_demand = 0
        self.demand_history = [0]

        self.last_day_cost = 0.0
        self.daily_costs = []

        # Start the daily review and demand processes.
        self.env.process(self.inventory_monitor())
        self.env.process(self.demand_generator())

    @property
    def inventory_level(self):
        """Return the current on-hand inventory level."""

        return self._inventory_level

    @inventory_level.setter
    def inventory_level(self, value):
        """
        Update the inventory level and record the duration spent at the
        previous level.
        """

        duration = (
            self.env.now
            - self.last_inventory_update_time
        )

        self.inventory_history[
            self.last_inventory_level
        ] += duration

        self._inventory_level = float(value)

        self.inventory_trajectory.append(
            (self.env.now, self._inventory_level)
        )

        self.last_inventory_update_time = self.env.now
        self.last_inventory_level = self._inventory_level

    @property
    def inventory_position(self):
        """
        Return the inventory position used by the (s, S) policy.
        """

        return self.inventory_level + self.on_order

    def _current_holding_cost(self):
        """
        Calculate holding cost accumulated since the latest inventory change.
        """

        duration = (
            self.env.now
            - self.last_inventory_update_time
        )

        return (
            max(self.inventory_level, 0)
            * duration
            * self.holding_cost
        )

    def _current_shortage_cost(self):
        """
        Calculate shortage cost accumulated since the latest inventory change.
        """

        duration = (
            self.env.now
            - self.last_inventory_update_time
        )

        return (
            max(-self.inventory_level, 0)
            * duration
            * self.shortage_cost
        )

    @property
    def total_holding_cost(self):
        """
        Return the total continuously accumulated holding cost.
        """

        historical_cost = sum(
            max(level, 0) * duration * self.holding_cost
            for level, duration in self.inventory_history.items()
        )

        return historical_cost + self._current_holding_cost()

    @property
    def total_shortage_cost(self):
        """
        Return the total continuously accumulated shortage cost.
        """

        historical_cost = sum(
            max(-level, 0) * duration * self.shortage_cost
            for level, duration in self.inventory_history.items()
        )

        return historical_cost + self._current_shortage_cost()

    @property
    def total_cost(self):
        """Return ordering, holding, and shortage costs combined."""

        return (
            self.total_order_cost
            + self.total_holding_cost
            + self.total_shortage_cost
        )

    @property
    def average_day_cost(self):
        """Return the average cost per simulated day."""

        if self.env.now <= 0:
            return 0.0

        return self.total_cost / self.env.now

    def inventory_monitor(self):
        """
        Review the inventory position once per day and apply the (s, S) rule.
        """

        while True:
            self.last_day_cost = self.total_cost

            yield self.env.timeout(
                self.inventory_check_interval
            )

            self.daily_costs.append(
                self.total_cost - self.last_day_cost
            )

            self.demand_history.append(
                self.last_step_demand
            )

            self.last_step_demand = 0

            if self.inventory_position < self.s_min:
                self.env.process(
                    self.order_up_to_s_max()
                )

    def order_up_to_s_max(self):
        """
        Place enough units to restore the inventory position to S.
        """

        quantity = (
            self.s_max
            - self.inventory_position
        )

        if quantity <= 0:
            return

        # The quantity is immediately added to the outstanding-order total.
        self.on_order += quantity

        # The setup cost is charged only for a positive order.
        self.total_order_cost += (
            self.order_setup_cost
            + self.order_incremental_cost * quantity
        )

        lead_time = random.uniform(
            self.lead_time_min,
            self.lead_time_max
        )

        yield self.env.timeout(lead_time)

        # The order is removed from the pipeline and added to inventory.
        self.inventory_level += quantity
        self.on_order -= quantity

    def demand_generator(self):
        """
        Generate demand arrivals using an exponential inter-arrival-time
        distribution and the product-specific demand-size distribution.
        """

        while True:
            inter_arrival_time = random.expovariate(
                self.demand_rate
            )

            yield self.env.timeout(
                inter_arrival_time
            )

            values, probabilities = (
                self.demand_distribution
            )

            demand_size = random.choices(
                values,
                weights=probabilities,
                k=1
            )[0]

            self.inventory_level -= demand_size
            self.last_step_demand += demand_size

    def plot_inventory_level(self):
        """Plot the continuous-time inventory trajectory."""

        times, levels = zip(
            *self.inventory_trajectory
        )

        plt.figure(figsize=(10, 4))

        plt.step(
            times,
            levels,
            where="post"
        )

        plt.fill_between(
            times,
            levels,
            step="post",
            alpha=0.2
        )

        plt.title("Inventory Level Over Time")
        plt.xlabel("Simulation Time")
        plt.ylabel("Inventory Level")
        plt.grid(alpha=0.2)
        plt.show()


# ============================================================================
# PRODUCT CONTROLLED BY A REINFORCEMENT LEARNING AGENT
# ============================================================================

class AgentProduct:
    """
    Simulate one product whose daily order quantity is selected by an agent.

    Action 0 means that no order is placed. A positive action represents the
    exact number of units to order.
    """

    def __init__(
        self,
        env: simpy.Environment,
        demand_rate: float = 0.1,
        demand_distribution=(
            [1, 2, 3, 4],
            [1/6, 1/3, 1/3, 1/6]
        ),
        inventory_level: float = 60,
        inventory_check_interval: float = 1,
        lead_time_min: float = 0.5,
        lead_time_max: float = 1.0,
        order_setup_cost: float = 10,
        order_incremental_cost: float = 3,
        holding_cost: float = 1,
        shortage_cost: float = 7
    ):
        self.env = env

        self.demand_rate = demand_rate
        self.demand_distribution = demand_distribution

        self._inventory_level = float(inventory_level)
        self.inventory_check_interval = inventory_check_interval

        self.lead_time_min = lead_time_min
        self.lead_time_max = lead_time_max

        self.order_setup_cost = order_setup_cost
        self.order_incremental_cost = order_incremental_cost
        self.holding_cost = holding_cost
        self.shortage_cost = shortage_cost

        # The action selected by the reinforcement learning agent.
        self.action = 0

        self.actions = list(
            range(ACTION_SIZE)
        )

        self.action_size = ACTION_SIZE

        # Total outstanding quantity.
        self.on_order = 0.0

        self.total_order_cost = 0.0

        self.inventory_history = defaultdict(float)

        self.inventory_trajectory = [
            (0.0, self._inventory_level)
        ]

        self.last_inventory_level = self._inventory_level
        self.last_inventory_update_time = 0.0

        self.last_day_cost = 0.0
        self.daily_costs = []

        self.env.process(self.inventory_monitor())
        self.env.process(self.demand_generator())

    @property
    def inventory_level(self):
        """Return the current on-hand inventory level."""

        return self._inventory_level

    @inventory_level.setter
    def inventory_level(self, value):
        """
        Update the inventory level and record the duration spent at the
        previous level.
        """

        duration = (
            self.env.now
            - self.last_inventory_update_time
        )

        self.inventory_history[
            self.last_inventory_level
        ] += duration

        self._inventory_level = float(value)

        self.inventory_trajectory.append(
            (self.env.now, self._inventory_level)
        )

        self.last_inventory_update_time = self.env.now
        self.last_inventory_level = self._inventory_level

    def _current_holding_cost(self):
        """Calculate the holding cost in the current inventory interval."""

        duration = (
            self.env.now
            - self.last_inventory_update_time
        )

        return (
            max(self.inventory_level, 0)
            * duration
            * self.holding_cost
        )

    def _current_shortage_cost(self):
        """Calculate the shortage cost in the current inventory interval."""

        duration = (
            self.env.now
            - self.last_inventory_update_time
        )

        return (
            max(-self.inventory_level, 0)
            * duration
            * self.shortage_cost
        )

    @property
    def total_holding_cost(self):
        """Return the continuously accumulated holding cost."""

        historical_cost = sum(
            max(level, 0) * duration * self.holding_cost
            for level, duration in self.inventory_history.items()
        )

        return historical_cost + self._current_holding_cost()

    @property
    def total_shortage_cost(self):
        """Return the continuously accumulated shortage cost."""

       historical_cost = sum(
            max(-level, 0) * duration * self.shortage_cost
            for level, duration in self.inventory_history.items()
        )

        return historical_cost + self._current_shortage_cost()

    @property
    def total_cost(self):
        """Return ordering, holding, and shortage costs combined."""

        return (
            self.total_order_cost
            + self.total_holding_cost
            + self.total_shortage_cost
        )

    @property
    def average_day_cost(self):
        """Return the average total cost per simulated day."""

        if self.env.now <= 0:
            return 0.0

        return self.total_cost / self.env.now

    def inventory_monitor(self):
        """
        Apply the action selected by the agent once per simulated day.
        """

        while True:
            self.last_day_cost = self.total_cost

            yield self.env.timeout(
                self.inventory_check_interval
            )

            self.daily_costs.append(
                self.total_cost - self.last_day_cost
            )

            self.place_order(self.action)

    def place_order(self, quantity):
        """
        Place a replenishment order for the specified quantity.

        A zero quantity means that no order is placed and no setup cost is
        charged.
        """

        quantity = int(quantity)

        if quantity <= 0:
            return

        self.on_order += quantity

        self.total_order_cost += (
            self.order_setup_cost
            + self.order_incremental_cost * quantity
        )

        self.env.process(
            self.receive_order(quantity)
        )

    def receive_order(self, quantity):
        """
        Deliver an outstanding order after a random product-specific lead time.
        """

        lead_time = random.uniform(
            self.lead_time_min,
            self.lead_time_max
        )

        yield self.env.timeout(lead_time)

        self.inventory_level += quantity
        self.on_order -= quantity

    def demand_generator(self):
        """
        Generate stochastic product demand in continuous simulation time.
        """

        while True:
            inter_arrival_time = random.expovariate(
                self.demand_rate
            )

            yield self.env.timeout(
                inter_arrival_time
            )

            values, probabilities = (
                self.demand_distribution
            )

            demand_size = random.choices(
                values,
                weights=probabilities,
                k=1
            )[0]

            self.inventory_level -= demand_size


# ============================================================================
# WAREHOUSE ENVIRONMENT USED BY THE RL AGENTS
# ============================================================================

class WarehouseEnv:
    """
    Simulate the complete two-product warehouse.

    State:
        [
            inventory_product_1,
            inventory_product_2,
            on_order_product_1,
            on_order_product_2
        ]

    Action:
        One integer encoding the two daily order quantities.

    Reward:
        Negative daily total cost.
    """

    def __init__(self, seed: int | None = None):
        self.seed = seed

        random.seed(seed)

        self.timer = 0
        self.env = simpy.Environment()

        self.action_size = ACTION_SIZE
        self.total_actions = TOTAL_ACTIONS

        self._create_products()

    def _create_products(self):
        """
        Create both products and initialize episode-level cost variables.
        """

        self.product_1 = AgentProduct(
            self.env,
            demand_rate=0.1,
            demand_distribution=(
                [1, 2, 3, 4],
                [1/6, 1/3, 1/3, 1/6]
            ),
            inventory_level=20,
            inventory_check_interval=1,
            lead_time_min=0.5,
            lead_time_max=1.0,
            order_setup_cost=10,
            order_incremental_cost=3,
            holding_cost=1,
            shortage_cost=7
        )

        self.product_2 = AgentProduct(
            self.env,
            demand_rate=0.1,
            demand_distribution=(
                [5, 4, 3, 2],
                [1/8, 1/2, 1/4, 1/8]
            ),
            inventory_level=20,
            inventory_check_interval=1,
            lead_time_min=0.2,
            lead_time_max=0.7,
            order_setup_cost=10,
            order_incremental_cost=3,
            holding_cost=1,
            shortage_cost=7
        )

        self.last_total_cost_1 = 0.0
        self.last_total_cost_2 = 0.0

        self.done = False

    def get_state(self):
        """Return the current four-dimensional warehouse state."""

        return [
            self.product_1.inventory_level,
            self.product_2.inventory_level,
            self.product_1.on_order,
            self.product_2.on_order
        ]

    def step(self, action: int):
        """
        Apply one combined action and advance the simulation by one day.
        """

        order_product_1, order_product_2 = action_to_pair(
            action,
            self.action_size
        )

        self.product_1.action = order_product_1
        self.product_2.action = order_product_2

        self.timer += 1

        self.env.run(
            until=self.timer
        )

        self.done = (
            self.timer >= NUM_STEPS
        )

        total_cost_1 = self.product_1.total_cost
        total_cost_2 = self.product_2.total_cost

        daily_cost_1 = (
            total_cost_1
            - self.last_total_cost_1
        )

        daily_cost_2 = (
            total_cost_2
            - self.last_total_cost_2
        )

        overall_cost = (
            daily_cost_1
            + daily_cost_2
        )

        # Reinforcement learning algorithms maximize reward.
        total_reward = -overall_cost

        self.last_total_cost_1 = total_cost_1
        self.last_total_cost_2 = total_cost_2

        return (
            self.get_state(),
            total_reward,
            self.done,
            self.product_1.total_holding_cost
            + self.product_2.total_holding_cost,
            self.product_1.total_shortage_cost
            + self.product_2.total_shortage_cost,
            self.product_1.total_order_cost
            + self.product_2.total_order_cost,
            overall_cost
        )

    def reset(self, seed: int | None = None):
        """
        Create a new simulation episode and return the initial state.
        """

        if seed is not None:
            self.seed = seed

        random.seed(self.seed)

        self.timer = 0
        self.env = simpy.Environment()

        self._create_products()

        return self.get_state()


# ============================================================================
# DOUBLE DQN NETWORK
# ============================================================================

class DQN(nn.Module):
    """
    Approximate one Q-value for each combined replenishment action.
    """

    def __init__(
        self,
        in_dim: int = 4,
        out_dim: int = TOTAL_ACTIONS
    ):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, out_dim)
        )

    def forward(self, state):
        """Return all action Q-values for the supplied states."""

        return self.model(state)


# ============================================================================
# STATE NORMALIZATION
# ============================================================================

def normalize_state(state):
    """
    Scale inventory and outstanding-order values before they enter a network.
    """

    return np.asarray(
        state,
        dtype=np.float32
    ) / STATE_SCALE


# ============================================================================
# EPSILON-GREEDY DQN ACTION SELECTION
# ============================================================================

def select_dqn_action(
    is_training,
    state,
    policy_net,
    epsilon,
    action_space_size=TOTAL_ACTIONS
):
    """
    Select a random action with probability epsilon.

    Otherwise, select the action with the largest predicted Q-value.
    """

    if (
        is_training
        and np.random.random() < epsilon
    ):
        return int(
            np.random.choice(
                action_space_size
            )
        )

    with torch.no_grad():
        normalized_state = normalize_state(
            state
        )

        state_tensor = torch.tensor(
            normalized_state,
            dtype=torch.float32
        ).unsqueeze(0)

        q_values = policy_net(
            state_tensor
        )

        return q_values.argmax(
            dim=1
        ).item()


# ============================================================================
# DOUBLE DQN OPTIMIZATION
# ============================================================================

def optimize_dqn(
    batch,
    policy_net,
    target_net,
    optimizer,
    gamma
):
    """
    Perform one Double DQN optimization step.

    The online policy network selects the best next action.
    The target network evaluates the selected next action.
    """

    states, actions, rewards, next_states, dones = zip(
        *batch
    )

    states = torch.tensor(
        np.asarray([
            normalize_state(state)
            for state in states
        ]),
        dtype=torch.float32
    )

    actions = torch.tensor(
        actions,
        dtype=torch.long
    ).unsqueeze(1)

    rewards = torch.tensor(
        rewards,
        dtype=torch.float32
    ).unsqueeze(1)

    next_states = torch.tensor(
        np.asarray([
            normalize_state(state)
            for state in next_states
        ]),
        dtype=torch.float32
    )

    dones = torch.tensor(
        dones,
        dtype=torch.float32
    ).unsqueeze(1)

    # Select Q(s, a) for the actions stored in the replay transitions.
    current_q_values = policy_net(
        states
    ).gather(
        1,
        actions
    )

    with torch.no_grad():
        # The online network chooses the next action.
        next_actions = policy_net(
            next_states
        ).argmax(
            dim=1,
            keepdim=True
        )

        # The target network evaluates that action.
        next_q_values = target_net(
            next_states
        ).gather(
            1,
            next_actions
        )

        target_q_values = (
            rewards
            + gamma
            * next_q_values
            * (1 - dones)
        )

    # Huber loss is more robust than MSE to large TD errors.
    loss = nn.functional.smooth_l1_loss(
        current_q_values,
        target_q_values
    )

    optimizer.zero_grad()
    loss.backward()

    # Limit the gradient norm to improve training stability.
    torch.nn.utils.clip_grad_norm_(
        policy_net.parameters(),
        max_norm=1.0
    )

    optimizer.step()

    return loss.item()


# ============================================================================
# DOUBLE DQN TRAINING
# ============================================================================

def train_dqn(
    seed=42,
    gamma=0.9,
    lr=0.0001,
    num_episodes=500,
    reward_scale=100.0
):
    """
    Train a Double DQN agent over multiple stochastic simulation episodes.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    policy_net = DQN(
        in_dim=4,
        out_dim=TOTAL_ACTIONS
    )

    target_net = DQN(
        in_dim=4,
        out_dim=TOTAL_ACTIONS
    )

    target_net.load_state_dict(
        policy_net.state_dict()
    )

    target_net.eval()

    optimizer = optim.Adam(
        policy_net.parameters(),
        lr=lr
    )

    # Experience replay configuration.
    replay_memory = deque(
        maxlen=50_000
    )

    batch_size = 256

    # Epsilon-greedy exploration configuration.
    epsilon_start = 1.0
    epsilon_end = 0.05
    epsilon_decay = 30_000

    # Copy the online parameters to the target network every 10 episodes.
    target_update = 10

    total_steps = 0

    episode_returns = []
    episode_epsilons = []
    episode_losses = []

    for episode in tqdm.tqdm(
        range(num_episodes)
    ):
        # A different simulation seed is used in each episode.
        episode_seed = seed + episode

        random.seed(episode_seed)
        np.random.seed(episode_seed)

        env = WarehouseEnv(
            episode_seed
        )

        state = env.reset(
            seed=episode_seed
        )

        total_reward = 0.0
        total_cost = 0.0
        losses = []

        while True:
            epsilon = (
                epsilon_end
                + (epsilon_start - epsilon_end)
                * np.exp(
                    -total_steps
                    / epsilon_decay
                )
            )

            action = select_dqn_action(
                is_training=True,
                state=state,
                policy_net=policy_net,
                epsilon=epsilon,
                action_space_size=TOTAL_ACTIONS
            )

            (
                next_state,
                reward,
                done,
                holding_cost,
                shortage_cost,
                order_cost,
                overall_cost
            ) = env.step(action)

            # Reward scaling changes the numerical magnitude but not the
            # relative preference between actions.
            scaled_reward = (
                reward / reward_scale
            )

            replay_memory.append(
                (
                    state,
                    action,
                    scaled_reward,
                    next_state,
                    done
                )
            )

            state = next_state

            total_reward += reward
            total_cost += overall_cost
            total_steps += 1

            if len(replay_memory) >= batch_size:
                batch = random.sample(
                    replay_memory,
                    batch_size
                )

                loss = optimize_dqn(
                    batch,
                    policy_net,
                    target_net,
                    optimizer,
                    gamma
                )

                losses.append(loss)

            if done:
                break

        if (
            episode + 1
        ) % target_update == 0:
            target_net.load_state_dict(
                policy_net.state_dict()
            )

        average_loss = (
            np.mean(losses)
            if losses
            else np.nan
        )

        episode_returns.append(
            total_reward
        )

        episode_epsilons.append(
            epsilon
        )

        episode_losses.append(
            average_loss
        )

        if (episode + 1) % 10 == 0:
            print(
                f"Episode {episode + 1}/{num_episodes} | "
                f"Reward: {total_reward:.2f} | "
                f"Holding: {holding_cost:.2f} | "
                f"Shortage: {shortage_cost:.2f} | "
                f"Order: {order_cost:.2f} | "
                f"Total cost: {total_cost:.2f} | "
                f"Epsilon: {epsilon:.3f} | "
                f"Loss: {average_loss:.5f}"
            )

    return (
        np.asarray(episode_returns),
        np.asarray(episode_epsilons),
        np.asarray(episode_losses),
        total_steps,
        policy_net
    )


# ============================================================================
# DOUBLE DQN EVALUATION
# ============================================================================

def evaluate_dqn(
    policy_net,
    test_seeds
):
    """
    Evaluate the trained DQN greedily on a fixed collection of seeds.
    """

    test_returns = []
    daily_costs = []
    holding_costs = []
    shortage_costs = []
    order_costs = []
    selected_actions = []

    policy_net.eval()

    with torch.no_grad():
        for episode_seed in tqdm.tqdm(
            test_seeds
        ):
            random.seed(episode_seed)
            np.random.seed(episode_seed)

            env = WarehouseEnv(
                episode_seed
            )

            state = env.reset(
                seed=episode_seed
            )

            total_reward = 0.0
            episode_actions = []

            while True:
                action = select_dqn_action(
                    is_training=False,
                    state=state,
                    policy_net=policy_net,
                    epsilon=0.0,
                    action_space_size=TOTAL_ACTIONS
                )

                order_pair = action_to_pair(
                    action,
                    ACTION_SIZE
                )

                episode_actions.append(
                    order_pair
                )

                (
                    next_state,
                    reward,
                    done,
                    holding_cost,
                    shortage_cost,
                    order_cost,
                    overall_cost
                ) = env.step(action)

                total_reward += reward
                state = next_state

                if done:
                    break

            test_returns.append(
                total_reward
            )

            holding_costs.append(
                env.product_1.total_holding_cost
                + env.product_2.total_holding_cost
            )

            shortage_costs.append(
                env.product_1.total_shortage_cost
                + env.product_2.total_shortage_cost
            )

            order_costs.append(
                env.product_1.total_order_cost
                + env.product_2.total_order_cost
            )

            daily_costs.append([
                cost_1 + cost_2
                for cost_1, cost_2 in zip(
                    env.product_1.daily_costs,
                    env.product_2.daily_costs
                )
            ])

            selected_actions.append(
                episode_actions
            )

    return (
        np.asarray(test_returns),
        np.asarray(daily_costs),
        np.asarray(holding_costs),
        np.asarray(shortage_costs),
        np.asarray(order_costs),
        np.asarray(selected_actions)
    )


# ============================================================================
# S-MIN/S-MAX BENCHMARK ENVIRONMENT
# ============================================================================

class SMinSMaxEnv:
    """
    Create the two products controlled by the benchmark (s, S) policy.
    """

    def __init__(self, seed=None):
        random.seed(seed)

        self.env = simpy.Environment()

        self.product_1 = Product(
            self.env,
            demand_rate=0.1,
            demand_distribution=(
                [1, 2, 3, 4],
                [1/6, 1/3, 1/3, 1/6]
            ),
            inventory_level=20,
            inventory_check_interval=1,
            lead_time_min=0.5,
            lead_time_max=1.0,
            s_max=20,
            s_min=0,
            order_setup_cost=10,
            order_incremental_cost=3,
            holding_cost=1,
            shortage_cost=7
        )

        self.product_2 = Product(
            self.env,
            demand_rate=0.1,
            demand_distribution=(
                [5, 4, 3, 2],
                [1/8, 1/2, 1/4, 1/8]
            ),
            inventory_level=20,
            inventory_check_interval=1,
            lead_time_min=0.2,
            lead_time_max=0.7,
            s_max=20,
            s_min=5,
            order_setup_cost=10,
            order_incremental_cost=3,
            holding_cost=1,
            shortage_cost=7
        )


def run_s_min_s_max(seed, until):
    """
    Run one replication of the benchmark policy.
    """

    warehouse = SMinSMaxEnv(
        seed
    )

    warehouse.env.run(
        until=until
    )

    return warehouse


def run_s_min_s_max_replications(
    test_seeds,
    until
):
    """
    Run the benchmark policy on all supplied evaluation seeds.
    """

    return [
        run_s_min_s_max(
            seed,
            until
        )
        for seed in test_seeds
    ]


# ============================================================================
# WELCH WARM-UP ANALYSIS
# ============================================================================

class Welch:
    """
    Estimate an initial transient period using a smoothed replication-mean
    process.
    """

    def __init__(
        self,
        process,
        window_size,
        tolerance
    ):
        self.process = np.asarray(
            process
        )

        self.window_size = window_size
        self.tolerance = tolerance

        self.replications_mean = np.mean(
            self.process,
            axis=0
        )

        self.averaged_process = (
            self._welch()
        )

        (
            self.differences,
            self.warmup_period
        ) = self._find_steady_state()

    @staticmethod
    def moving_average(
        array,
        window_size
    ):
        """Return the moving average of a one-dimensional array."""

        weights = (
            np.ones(window_size)
            / window_size
        )

        return np.convolve(
            array,
            weights,
            mode="valid"
        )

    def _welch(self):
        """
        Smooth the replication-mean process using a centered moving window.
        """

        averaged_process = []

        for index in range(
            1,
            self.replications_mean.shape[0]
            - self.window_size
        ):
            if index <= self.window_size:
                averaged_process.append(
                    self.replications_mean[
                        :2 * index - 1
                    ].mean()
                )
            else:
                half_window = (
                    self.window_size // 2
                )

                averaged_process.append(
                    self.replications_mean[
                        index - half_window:
                        index + half_window
                    ].mean()
                )

        return np.asarray(
            averaged_process
        )

    def _find_steady_state(self):
        """
        Find the first point where the absolute smoothed change is below the
        selected tolerance.
        """

        averaged = self.moving_average(
            self.averaged_process,
            self.window_size
        )

        differences = np.diff(
            averaged.flatten()
        )

        for index, difference in enumerate(
            differences
        ):
            if abs(difference) < self.tolerance:
                return (
                    differences,
                    index + self.window_size
                )

        return differences, 0

    def plot(self):
        """Plot the smoothed process and estimated warm-up period."""

        plt.figure(figsize=(10, 4))

        plt.plot(
            self.averaged_process,
            label="Smoothed Replication Mean"
        )

        plt.axvline(
            self.warmup_period,
            color="red",
            linestyle="--",
            label=f"Estimated Warm-Up: {self.warmup_period}"
        )

        plt.title("Welch Warm-Up Analysis")
        plt.xlabel("Simulation Day")
        plt.ylabel("Smoothed Daily Cost")
        plt.legend()
        plt.grid(alpha=0.2)
        plt.show()


# ============================================================================
# STATISTICAL ANALYSIS
# ============================================================================

def percent_point_function(
    alpha,
    number_of_replications
):
    """
    Return the critical value for a two-sided Student's t confidence interval.
    """

    return stats.t.ppf(
        1 - alpha / 2,
        number_of_replications - 1
    )


def analyze_costs_from_array(
    runs,
    warmup_period,
    alpha=0.05
):
    """
    Estimate the mean daily cost, between-replication variance, and confidence
    interval half-width after removing the warm-up period.
    """

    runs = np.asarray(
        runs
    )

    replication_means = [
        statistics.mean(
            run[warmup_period:]
        )
        for run in runs
    ]

    number_of_replications = len(
        replication_means
    )

    sample_mean = statistics.mean(
        replication_means
    )

    sample_variance = statistics.variance(
        replication_means
    )

    critical_value = percent_point_function(
        alpha,
        number_of_replications
    )

    half_interval = critical_value * np.sqrt(
        sample_variance
        / number_of_replications
    )

    return (
        sample_mean,
        sample_variance,
        half_interval
    )


def compare(
    reference_daily_costs,
    agent_daily_costs,
    alpha=0.05
):
    """
    Perform a paired comparison of average daily costs.

    The reported difference is:

        benchmark cost - agent cost
    """

    reference_means = np.asarray(
        reference_daily_costs
    ).mean(axis=1)

    agent_means = np.asarray(
        agent_daily_costs
    ).mean(axis=1)

    differences = (
        reference_means
        - agent_means
    )

    print_confidence_difference(
        differences,
        alpha
    )


def compare_components(
    reference_costs,
    agent_costs,
    alpha=0.05
):
    """
    Perform a paired comparison of cumulative cost components.
    """

    differences = (
        np.asarray(reference_costs)
        - np.asarray(agent_costs)
    )

    print_confidence_difference(
        differences,
        alpha
    )


def print_confidence_difference(
    differences,
    alpha=0.05
):
    """
    Print the mean paired difference and its two-sided confidence interval.
    """

    differences = np.asarray(
        differences,
        dtype=np.float64
    )

    number_of_replications = len(
        differences
    )

    difference_mean = differences.mean()

    difference_variance = differences.var(
        ddof=1
    )

    critical_value = percent_point_function(
        alpha,
        number_of_replications
    )

    half_interval = critical_value * np.sqrt(
        difference_variance
        / number_of_replications
    )

    print(
        f"Point estimate: {difference_mean:.2f} "
        f"[{difference_mean - half_interval:.2f}; "
        f"{difference_mean + half_interval:.2f}]"
    )

In [ ]:
# ============================================================================
# DOUBLE DQN TRAINING
# ============================================================================

print("Training the Double DQN agent...")

(
    dqn_returns,
    dqn_epsilons,
    dqn_losses,
    dqn_steps_done,
    dqn_policy_net
) = train_dqn(
    seed=7,
    gamma=0.90,
    lr=0.0001,
    num_episodes=500,
    reward_scale=100.0
)


# ============================================================================
# DOUBLE DQN EVALUATION
# ============================================================================

test_seeds = seeds[20:40]

print("\nEvaluating the Double DQN agent...")

(
    dqn_test_returns,
    dqn_daily_costs,
    dqn_holding_costs,
    dqn_shortage_costs,
    dqn_order_costs,
    dqn_actions
) = evaluate_dqn(
    dqn_policy_net,
    test_seeds=test_seeds
)


# ============================================================================
# S-MIN/S-MAX BENCHMARK EVALUATION
# ============================================================================

print("\nEvaluating the S-min/S-max policy...")

warehouse_runs = run_s_min_s_max_replications(
    test_seeds=test_seeds,
    until=NUM_STEPS
)

# Combine the daily costs of the two products for each replication.
s_policy_daily_costs = np.asarray([
    [
        cost_1 + cost_2
        for cost_1, cost_2 in zip(
            run.product_1.daily_costs,
            run.product_2.daily_costs
        )
    ]
    for run in warehouse_runs
])

s_policy_holding_costs = np.asarray([
    run.product_1.total_holding_cost
    + run.product_2.total_holding_cost
    for run in warehouse_runs
])

s_policy_shortage_costs = np.asarray([
    run.product_1.total_shortage_cost
    + run.product_2.total_shortage_cost
    for run in warehouse_runs
])

s_policy_order_costs = np.asarray([
    run.product_1.total_order_cost
    + run.product_2.total_order_cost
    for run in warehouse_runs
])


# ============================================================================
# WELCH WARM-UP ANALYSIS
# ============================================================================

welch = Welch(
    s_policy_daily_costs,
    window_size=15,
    tolerance=0.01
)

welch.plot()


# ============================================================================
# S-MIN/S-MAX STATISTICAL RESULTS
# ============================================================================

alpha = 0.05

(
    s_daily_mean,
    s_daily_variance,
    s_half_interval
) = analyze_costs_from_array(
    s_policy_daily_costs,
    warmup_period=welch.warmup_period,
    alpha=alpha
)

print("\nS-min/S-max Policy Results:")

print(
    f"Daily Cost Sample Mean: "
    f"{s_daily_mean:.2f}"
)

print(
    f"Daily Cost Sample Variance: "
    f"{s_daily_variance:.2f}"
)

print(
    f"Confidence Interval Half-Width: "
    f"{s_half_interval:.2f}"
)

print(
    f"95% Confidence Interval: "
    f"({s_daily_mean - s_half_interval:.2f}, "
    f"{s_daily_mean + s_half_interval:.2f})"
)

print(
    f"Relative Error: "
    f"{100 * s_half_interval / s_daily_mean:.2f}%"
)


# ============================================================================
# PAIRED COMPARISON: S-MIN/S-MAX MINUS DOUBLE DQN
# ============================================================================

print("\nComparison Between S-min/S-max and Double DQN")
print("Reported difference: S-min/S-max minus Double DQN")

print("\nAverage Daily Cost:")
compare(
    s_policy_daily_costs,
    dqn_daily_costs
)

print("\nCumulative Holding Cost:")
compare_components(
    s_policy_holding_costs,
    dqn_holding_costs
)

print("\nCumulative Shortage Cost:")
compare_components(
    s_policy_shortage_costs,
    dqn_shortage_costs
)

print("\nCumulative Ordering Cost:")
compare_components(
    s_policy_order_costs,
    dqn_order_costs
)


# ============================================================================
# DOUBLE DQN ACTION ANALYSIS
# ============================================================================

flat_dqn_actions = dqn_actions.reshape(
    -1,
    2
)

print("\nDouble DQN Action Analysis:")

print(
    f"Average Order for Product 1: "
    f"{flat_dqn_actions[:, 0].mean():.2f}"
)

print(
    f"Average Order for Product 2: "
    f"{flat_dqn_actions[:, 1].mean():.2f}"
)

print(
    f"Maximum Order for Product 1: "
    f"{flat_dqn_actions[:, 0].max()}"
)

print(
    f"Maximum Order for Product 2: "
    f"{flat_dqn_actions[:, 1].max()}"
)

print(
    f"No-Order Percentage for Product 1: "
    f"{100 * np.mean(flat_dqn_actions[:, 0] == 0):.2f}%"
)

print(
    f"No-Order Percentage for Product 2: "
    f"{100 * np.mean(flat_dqn_actions[:, 1] == 0):.2f}%"
)


# ============================================================================
# PLOT 1: ALL COLORED REPLICATIONS
# ============================================================================

plt.figure(figsize=(12, 6))

number_of_replications = len(
    test_seeds
)

# Different shades of blue are assigned to DQN replications.
dqn_colors = plt.cm.Blues(
    np.linspace(
        0.35,
        0.95,
        number_of_replications
    )
)

# Different shades of orange are assigned to benchmark replications.
benchmark_colors = plt.cm.Oranges(
    np.linspace(
        0.35,
        0.95,
        number_of_replications
    )
)

for replication in range(
    number_of_replications
):
    plt.plot(
        dqn_daily_costs[replication],
        color=dqn_colors[replication],
        alpha=0.65,
        linewidth=1
    )

    plt.plot(
        s_policy_daily_costs[replication],
        color=benchmark_colors[replication],
        alpha=0.65,
        linewidth=1,
        linestyle="--"
    )

# Create a concise legend instead of one entry for every replication.
legend_handles = [
    plt.Line2D(
        [0],
        [0],
        color="tab:blue",
        linewidth=2,
        label="Double DQN Replications"
    ),
    plt.Line2D(
        [0],
        [0],
        color="tab:orange",
        linewidth=2,
        linestyle="--",
        label="S-min/S-max Replications"
    )
]

plt.title(
    "Daily Costs for All Replications: "
    "Double DQN vs S-min/S-max"
)

plt.xlabel("Simulation Day")
plt.ylabel("Daily Cost")
plt.legend(handles=legend_handles)
plt.grid(alpha=0.2)
plt.show()


# ============================================================================
# PLOT 2: AVERAGE DAILY COST TRAJECTORIES
# ============================================================================

plt.figure(figsize=(12, 5))

plt.plot(
    dqn_daily_costs.mean(axis=0),
    label="Double DQN Agent",
    linewidth=2
)

plt.plot(
    s_policy_daily_costs.mean(axis=0),
    label="S-min/S-max Policy",
    linewidth=2
)

plt.title(
    "Average Daily Costs: "
    "Double DQN vs S-min/S-max"
)

plt.xlabel("Simulation Day")
plt.ylabel("Average Daily Cost")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# ============================================================================
# PLOT 3: DOUBLE DQN TRAINING RETURNS
# ============================================================================

plt.figure(figsize=(10, 4))

plt.plot(
    dqn_returns
)

plt.title(
    "Double DQN Training Returns"
)

plt.xlabel("Episode")
plt.ylabel("Total Unscaled Reward")
plt.grid(alpha=0.2)
plt.show()


# ============================================================================
# PLOT 4: EPSILON DECAY
# ============================================================================

plt.figure(figsize=(10, 4))

plt.plot(
    dqn_epsilons
)

plt.title(
    "Double DQN Exploration Rate"
)

plt.xlabel("Episode")
plt.ylabel("Epsilon")
plt.grid(alpha=0.2)
plt.show()


# ============================================================================
# PLOT 5: DOUBLE DQN TRAINING LOSS
# ============================================================================

plt.figure(figsize=(10, 4))

plt.plot(
    dqn_losses
)

plt.title(
    "Double DQN Training Loss"
)

plt.xlabel("Episode")
plt.ylabel("Average Huber Loss")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# ============================================================================
# ACTOR-CRITIC NETWORK
# ============================================================================

class ActorCriticNet(nn.Module):
    """
    Actor-Critic network with a shared feature extractor.

    The actor produces a categorical distribution over all combined actions.
    The critic estimates the value of the current warehouse state.
    """

    def __init__(
        self,
        in_dim=4,
        action_dim=TOTAL_ACTIONS
    ):
        super().__init__()

        self.shared_network = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU()
        )

        self.actor_head = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

        self.critic_head = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, state):
        """
        Return action logits and the estimated state value.
        """

        features = self.shared_network(
            state
        )

        action_logits = self.actor_head(
            features
        )

        state_value = self.critic_head(
            features
        )

        return action_logits, state_value


# ============================================================================
# ACTOR-CRITIC ACTION SELECTION
# ============================================================================

def select_actor_critic_action(
    model,
    state,
    deterministic=False
):
    """
    Select an action from the actor's categorical distribution.

    During training, an action is sampled.
    During deterministic evaluation, the highest-logit action is selected.
    """

    normalized_state = normalize_state(
        state
    )

    state_tensor = torch.tensor(
        normalized_state,
        dtype=torch.float32
    ).unsqueeze(0)

    logits, value = model(
        state_tensor
    )

    action_distribution = Categorical(
        logits=logits
    )

    if deterministic:
        action = logits.argmax(
            dim=1
        )
    else:
        action = action_distribution.sample()

    log_probability = (
        action_distribution.log_prob(action)
    )

    entropy = (
        action_distribution.entropy()
    )

    return (
        action.item(),
        log_probability,
        entropy,
        value.squeeze()
    )


# ============================================================================
# ACTOR-CRITIC TRAINING
# ============================================================================

def train_actor_critic(
    seed=1,
    num_episodes=600,
    gamma=0.90,
    lr=0.0001,
    entropy_coefficient=0.01,
    value_coefficient=0.5,
    reward_scale=1000.0,
    print_every=10
):
    """
    Train the Actor-Critic model using one update after each complete episode.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = ActorCriticNet(
        in_dim=4,
        action_dim=TOTAL_ACTIONS
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr
    )

    episode_returns = []
    episode_costs = []
    actor_losses = []
    critic_losses = []
    entropy_values = []

    for episode in tqdm.tqdm(
        range(num_episodes)
    ):
        episode_seed = seed + episode

        random.seed(episode_seed)
        np.random.seed(episode_seed)

        env = WarehouseEnv(
            episode_seed
        )

        state = env.reset(
            seed=episode_seed
        )

        log_probabilities = []
        estimated_values = []
        scaled_rewards = []
        entropies = []

        total_reward = 0.0
        total_cost = 0.0

        while True:
            (
                action,
                log_probability,
                entropy,
                estimated_value
            ) = select_actor_critic_action(
                model,
                state,
                deterministic=False
            )

            (
                next_state,
                reward,
                done,
                holding_cost,
                shortage_cost,
                order_cost,
                overall_cost
            ) = env.step(action)

            scaled_reward = (
                reward / reward_scale
            )

            log_probabilities.append(
                log_probability
            )

            estimated_values.append(
                estimated_value
            )

            scaled_rewards.append(
                scaled_reward
            )

            entropies.append(
                entropy
            )

            total_reward += reward
            total_cost += overall_cost

            state = next_state

            if done:
                break

        # Calculate discounted returns backward through the episode.
        discounted_returns = []
        return_value = 0.0

        for reward in reversed(
            scaled_rewards
        ):
            return_value = (
                reward
                + gamma * return_value
            )

            discounted_returns.insert(
                0,
                return_value
            )

        discounted_returns = torch.tensor(
            discounted_returns,
            dtype=torch.float32
        )

        estimated_values = torch.stack(
            estimated_values
        )

        log_probabilities = torch.stack(
            log_probabilities
        )

        entropies = torch.stack(
            entropies
        )

        # Return normalization improves numerical stability.
        if len(discounted_returns) > 1:
            discounted_returns = (
                discounted_returns
                - discounted_returns.mean()
            ) / (
                discounted_returns.std()
                + 1e-8
            )

        # The advantage compares the observed return with the critic estimate.
        advantages = (
            discounted_returns
            - estimated_values.detach()
        )

        actor_loss = -(
            log_probabilities
            * advantages
        ).mean()

        critic_loss = (
            nn.functional.smooth_l1_loss(
                estimated_values,
                discounted_returns
            )
        )

        entropy_bonus = (
            entropies.mean()
        )

        total_loss = (
            actor_loss
            + value_coefficient * critic_loss
            - entropy_coefficient * entropy_bonus
        )

        optimizer.zero_grad()
        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        episode_returns.append(
            total_reward
        )

        episode_costs.append(
            total_cost
        )

        actor_losses.append(
            actor_loss.item()
        )

        critic_losses.append(
            critic_loss.item()
        )

        entropy_values.append(
            entropy_bonus.item()
        )

        if (
            episode + 1
        ) % print_every == 0:
            print(
                f"Episode {episode + 1}/{num_episodes} | "
                f"Reward: {total_reward:.2f} | "
                f"Holding: {holding_cost:.2f} | "
                f"Shortage: {shortage_cost:.2f} | "
                f"Order: {order_cost:.2f} | "
                f"Total cost: {total_cost:.2f} | "
                f"Actor loss: {actor_loss.item():.4f} | "
                f"Critic loss: {critic_loss.item():.4f}"
            )

    return (
        model,
        np.asarray(episode_returns),
        np.asarray(episode_costs),
        np.asarray(actor_losses),
        np.asarray(critic_losses),
        np.asarray(entropy_values)
    )


# ============================================================================
# ACTOR-CRITIC EVALUATION
# ============================================================================

def evaluate_actor_critic(
    model,
    test_seeds
):
    """
    Evaluate the Actor-Critic model deterministically on fixed test seeds.
    """

    test_returns = []
    daily_costs = []
    holding_costs = []
    shortage_costs = []
    order_costs = []
    selected_actions = []

    model.eval()

    with torch.no_grad():
        for episode_seed in tqdm.tqdm(
            test_seeds
        ):
            random.seed(episode_seed)
            np.random.seed(episode_seed)

            env = WarehouseEnv(
                episode_seed
            )

            state = env.reset(
                seed=episode_seed
            )

            total_reward = 0.0
            episode_actions = []

            while True:
                (
                    action,
                    _,
                    _,
                    _
                ) = select_actor_critic_action(
                    model,
                    state,
                    deterministic=True
                )

                order_pair = action_to_pair(
                    action,
                    ACTION_SIZE
                )

                episode_actions.append(
                    order_pair
                )

                (
                    next_state,
                    reward,
                    done,
                    holding_cost,
                    shortage_cost,
                    order_cost,
                    overall_cost
                ) = env.step(action)

                total_reward += reward
                state = next_state

                if done:
                    break

            test_returns.append(
                total_reward
            )

            holding_costs.append(
                env.product_1.total_holding_cost
                + env.product_2.total_holding_cost
            )

            shortage_costs.append(
                env.product_1.total_shortage_cost
                + env.product_2.total_shortage_cost
            )

            order_costs.append(
                env.product_1.total_order_cost
                + env.product_2.total_order_cost
            )

            daily_costs.append([
                cost_1 + cost_2
                for cost_1, cost_2 in zip(
                    env.product_1.daily_costs,
                    env.product_2.daily_costs
                )
            ])

            selected_actions.append(
                episode_actions
            )

    return (
        np.asarray(test_returns),
        np.asarray(daily_costs),
        np.asarray(holding_costs),
        np.asarray(shortage_costs),
        np.asarray(order_costs),
        np.asarray(selected_actions)
    )


# ============================================================================
# ACTOR-CRITIC TRAINING
# ============================================================================

print("Training the Actor-Critic agent...")

(
    actor_critic_model,
    actor_critic_returns,
    actor_critic_costs,
    actor_losses,
    critic_losses,
    actor_critic_entropies
) = train_actor_critic(
    seed=1,
    num_episodes=600,
    gamma=0.90,
    lr=0.0001,
    entropy_coefficient=0.01,
    value_coefficient=0.5,
    reward_scale=1000.0,
    print_every=10
)


# ============================================================================
# ACTOR-CRITIC EVALUATION
# ============================================================================

test_seeds = seeds[20:40]

print("\nEvaluating the Actor-Critic agent...")

(
    actor_critic_test_returns,
    actor_critic_daily_costs,
    actor_critic_holding_costs,
    actor_critic_shortage_costs,
    actor_critic_order_costs,
    actor_critic_actions
) = evaluate_actor_critic(
    actor_critic_model,
    test_seeds=test_seeds
)


# ============================================================================
# S-MIN/S-MAX BENCHMARK EVALUATION
# ============================================================================

print("\nEvaluating the S-min/S-max policy...")

warehouse_runs = run_s_min_s_max_replications(
    test_seeds=test_seeds,
    until=NUM_STEPS
)

s_policy_daily_costs = np.asarray([
    [
        cost_1 + cost_2
        for cost_1, cost_2 in zip(
            run.product_1.daily_costs,
            run.product_2.daily_costs
        )
    ]
    for run in warehouse_runs
])

s_policy_holding_costs = np.asarray([
    run.product_1.total_holding_cost
    + run.product_2.total_holding_cost
    for run in warehouse_runs
])

s_policy_shortage_costs = np.asarray([
    run.product_1.total_shortage_cost
    + run.product_2.total_shortage_cost
    for run in warehouse_runs
])

s_policy_order_costs = np.asarray([
    run.product_1.total_order_cost
    + run.product_2.total_order_cost
    for run in warehouse_runs
])


# ============================================================================
# WELCH WARM-UP ANALYSIS
# ============================================================================

welch_actor_critic = Welch(
    s_policy_daily_costs,
    window_size=15,
    tolerance=0.01
)

welch_actor_critic.plot()


# ============================================================================
# S-MIN/S-MAX STATISTICAL RESULTS
# ============================================================================

alpha = 0.05

(
    s_daily_mean,
    s_daily_variance,
    s_half_interval
) = analyze_costs_from_array(
    s_policy_daily_costs,
    warmup_period=welch_actor_critic.warmup_period,
    alpha=alpha
)

print("\nS-min/S-max Policy Results:")

print(
    f"Daily Cost Sample Mean: "
    f"{s_daily_mean:.2f}"
)

print(
    f"Daily Cost Sample Variance: "
    f"{s_daily_variance:.2f}"
)

print(
    f"Confidence Interval Half-Width: "
    f"{s_half_interval:.2f}"
)

print(
    f"95% Confidence Interval: "
    f"({s_daily_mean - s_half_interval:.2f}, "
    f"{s_daily_mean + s_half_interval:.2f})"
)

print(
    f"Relative Error: "
    f"{100 * s_half_interval / s_daily_mean:.2f}%"
)


# ============================================================================
# PAIRED COMPARISON: S-MIN/S-MAX MINUS ACTOR-CRITIC
# ============================================================================

print("\nComparison Between S-min/S-max and Actor-Critic")
print("Reported difference: S-min/S-max minus Actor-Critic")

print("\nAverage Daily Cost:")
compare(
    s_policy_daily_costs,
    actor_critic_daily_costs
)

print("\nCumulative Holding Cost:")
compare_components(
    s_policy_holding_costs,
    actor_critic_holding_costs
)

print("\nCumulative Shortage Cost:")
compare_components(
    s_policy_shortage_costs,
    actor_critic_shortage_costs
)

print("\nCumulative Ordering Cost:")
compare_components(
    s_policy_order_costs,
    actor_critic_order_costs
)


# ============================================================================
# ACTOR-CRITIC ACTION ANALYSIS
# ============================================================================

flat_actor_critic_actions = (
    actor_critic_actions.reshape(-1, 2)
)

print("\nActor-Critic Action Analysis:")

print(
    f"Average Order for Product 1: "
    f"{flat_actor_critic_actions[:, 0].mean():.2f}"
)

print(
    f"Average Order for Product 2: "
    f"{flat_actor_critic_actions[:, 1].mean():.2f}"
)

print(
    f"Maximum Order for Product 1: "
    f"{flat_actor_critic_actions[:, 0].max()}"
)

print(
    f"Maximum Order for Product 2: "
    f"{flat_actor_critic_actions[:, 1].max()}"
)

print(
    f"No-Order Percentage for Product 1: "
    f"{100 * np.mean(flat_actor_critic_actions[:, 0] == 0):.2f}%"
)

print(
    f"No-Order Percentage for Product 2: "
    f"{100 * np.mean(flat_actor_critic_actions[:, 1] == 0):.2f}%"
)


# ============================================================================
# PLOT 1: ALL COLORED REPLICATIONS
# ============================================================================

plt.figure(figsize=(12, 6))

number_of_replications = len(
    test_seeds
)

# Different shades of green are assigned to Actor-Critic replications.
actor_critic_colors = plt.cm.Greens(
    np.linspace(
        0.35,
        0.95,
        number_of_replications
    )
)

# Different shades of orange are assigned to benchmark replications.
benchmark_colors = plt.cm.Oranges(
    np.linspace(
        0.35,
        0.95,
        number_of_replications
    )
)

for replication in range(
    number_of_replications
):
    plt.plot(
        actor_critic_daily_costs[replication],
        color=actor_critic_colors[replication],
        alpha=0.65,
        linewidth=1
    )

    plt.plot(
        s_policy_daily_costs[replication],
        color=benchmark_colors[replication],
        alpha=0.65,
        linewidth=1,
        linestyle="--"
    )

legend_handles = [
    plt.Line2D(
        [0],
        [0],
        color="tab:green",
        linewidth=2,
        label="Actor-Critic Replications"
    ),
    plt.Line2D(
        [0],
        [0],
        color="tab:orange",
        linewidth=2,
        linestyle="--",
        label="S-min/S-max Replications"
    )
]

plt.title(
    "Daily Costs for All Replications: "
    "Actor-Critic vs S-min/S-max"
)

plt.xlabel("Simulation Day")
plt.ylabel("Daily Cost")
plt.legend(handles=legend_handles)
plt.grid(alpha=0.2)
plt.show()


# ============================================================================
# PLOT 2: AVERAGE DAILY COST TRAJECTORIES
# ============================================================================

plt.figure(figsize=(12, 5))

plt.plot(
    actor_critic_daily_costs.mean(axis=0),
    label="Actor-Critic Agent",
    linewidth=2
)

plt.plot(
    s_policy_daily_costs.mean(axis=0),
    label="S-min/S-max Policy",
    linewidth=2
)

plt.title(
    "Average Daily Costs: "
    "Actor-Critic vs S-min/S-max"
)

plt.xlabel("Simulation Day")
plt.ylabel("Average Daily Cost")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# ============================================================================
# PLOT 3: ACTOR-CRITIC TRAINING RETURNS
# ============================================================================

plt.figure(figsize=(10, 4))

plt.plot(
    actor_critic_returns
)

plt.title(
    "Actor-Critic Training Returns"
)

plt.xlabel("Episode")
plt.ylabel("Total Unscaled Reward")
plt.grid(alpha=0.2)
plt.show()


# ============================================================================
# PLOT 4: ACTOR AND CRITIC LOSSES
# ============================================================================

plt.figure(figsize=(10, 4))

plt.plot(
    actor_losses,
    label="Actor Loss"
)

plt.plot(
    critic_losses,
    label="Critic Loss"
)

plt.title(
    "Actor-Critic Training Losses"
)

plt.xlabel("Episode")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# ============================================================================
# PLOT 5: POLICY ENTROPY
# ============================================================================

plt.figure(figsize=(10, 4))

plt.plot(
    actor_critic_entropies
)

plt.title(
    "Actor-Critic Policy Entropy"
)

plt.xlabel("Episode")
plt.ylabel("Mean Entropy")
plt.grid(alpha=0.2)
plt.show()